# Chaînes 10 — Le groupe des rotations comme automorphismes des directions dérivées

Chapitre 49 du compendium *Chain Theory* (Wissner-Gross, 2026, preprint non relu, lu depuis la copie archivée au gisement), section 53. Deuxième chapitre de la Partie IV : la coordonnée d'ordre lue au chapitre précédent comme un compte, voici les rotations. Le patron de la série s'applique tel quel : **Mesuré** = ce que ce carnet énumère et vérifie lui-même (comptes de groupes finis, constantes de structure exactes, commutants résolus) ; **Cité** = la chaîne dérivative de la source (l'action sur les directions dérivées, le census des 102 bouts qubit) ; **Ouvert** = ce que la source laisse ouvert, ligne à ligne.

## Le cadre : après le compte, les rotations

« With order read as a count, the next chapter turns to the rotations. » Le chapitre 48 avait clos sur cette phrase. Le groupe des rotations n'est **pas posé** ici : il est *lu* comme les automorphismes des directions dérivées — le groupe que la littérature appelle SO(3) apparaît comme les symétries d'un ensemble fini de directions que la construction dérive elle-même. Trois objets finis portent le chapitre : le groupe d'identification au bout qubit (32 matrices monomiales à phases racines quatrièmes), son image sur les directions (le groupe diédral d'ordre 8), et le réseau des 24 rotations du cube où cette image s'incarne à indice trois.

## §53 — Théorème 53.1 (conditionnel) : le groupe des rotations

**Énoncé (traduit).** Le groupe des rotations de la théorie est le stabilisateur de l'unité hermitienne dans les isométries de la scission hermitienne de la trace — une algèbre de Lie de dimension trois, engendrée par les matrices *ad* des trois directions dérivées, aux crochets du symbole alterné ; son groupe connexe, les exponentielles aux points rationnels exacts, relevé au bout qubit par conjugaison par ses unitaires ; le relèvement fini, l'image diédrale d'ordre huit dans les vingt-quatre rotations du réseau. Chaque bout qubit des treize corps de la construction y est typé — proprement au bout proche d'un dock, improprement au bout loin d'un dock qubit-qubit —, les bouts qubit des docks sont spatiaux et les copies de hub internes.

**Standing : théorème conditionnel.** Prémisses : l'axiome, les six définitions du chapitre 1, la table Rosetta complète (chapitre I.1). Utilisé par : Open 17.1, Thm 54.1, Thm 57.1 et Open 57.2, Thm 59.1, Open 123.1, Thm 124.1.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Arithmetique exacte dans Q(i) : chaque complexe est une paire de rationnels reduits.
// Le rationnel porte des BigInteger (System.Numerics) ; aucune equality flottante ici.
public sealed class Rat : IEquatable<Rat>
{
    public BigInteger N { get; }
    public BigInteger D { get; }

    public Rat(BigInteger n, BigInteger d)
    {
        if (d.IsZero) throw new DivideByZeroException("Denominateur nul.");
        if (d.Sign < 0) { n = -n; d = -d; }
        BigInteger g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(n), d);
        if (!g.IsOne && !g.IsZero) { n /= g; d /= g; }
        N = n;
        D = d;
    }

    public static readonly Rat Zero = new(BigInteger.Zero, BigInteger.One);
    public static readonly Rat One = new(BigInteger.One, BigInteger.One);

    public static Rat FromInt(long k) => new(new BigInteger(k), BigInteger.One);

    public static Rat operator +(Rat a, Rat b) => new(a.N * b.D + b.N * a.D, a.D * b.D);
    public static Rat operator -(Rat a, Rat b) => new(a.N * b.D - b.N * a.D, a.D * b.D);
    public static Rat operator *(Rat a, Rat b) => new(a.N * b.N, a.D * b.D);
    public static Rat operator /(Rat a, Rat b) => new(a.N * b.D, a.D * b.N);
    public static Rat operator -(Rat a) => new(-a.N, a.D);

    public bool Equals(Rat? other) => other is not null && N == other.N && D == other.D;
    public override bool Equals(object? obj) => obj is Rat r && Equals(r);
    public override int GetHashCode() => HashCode.Combine(N, D);
    public static bool operator ==(Rat? a, Rat? b) => a is null ? b is null : a.Equals(b);
    public static bool operator !=(Rat? a, Rat? b) => !(a == b);

    public override string ToString() => D.IsOne
        ? N.ToString(CultureInfo.InvariantCulture)
        : N.ToString(CultureInfo.InvariantCulture) + "/" + D.ToString(CultureInfo.InvariantCulture);
}

public sealed class Cx : IEquatable<Cx>
{
    public Rat Re { get; }
    public Rat Im { get; }

    public Cx(Rat re, Rat im) { Re = re; Im = im; }

    public static readonly Cx Zero = new(Rat.Zero, Rat.Zero);
    public static readonly Cx One = new(Rat.One, Rat.Zero);
    public static readonly Cx I = new(Rat.Zero, Rat.One);

    public static Cx operator +(Cx a, Cx b) => new(a.Re + b.Re, a.Im + b.Im);
    public static Cx operator -(Cx a, Cx b) => new(a.Re - b.Re, a.Im - b.Im);
    public static Cx operator *(Cx a, Cx b) => new(a.Re * b.Re - a.Im * b.Im, a.Re * b.Im + a.Im * b.Re);
    public static Cx operator -(Cx a) => new(-a.Re, -a.Im);
    public static Cx Scale(Cx a, Rat k) => new(a.Re * k, a.Im * k);

    public bool Equals(Cx? other) => other is not null && Re == other.Re && Im == other.Im;
    public override bool Equals(object? obj) => obj is Cx c && Equals(c);
    public override int GetHashCode() => HashCode.Combine(Re, Im);
    public static bool operator ==(Cx? a, Cx? b) => a is null ? b is null : a.Equals(b);
    public static bool operator !=(Cx? a, Cx? b) => !(a == b);

    public override string ToString() => "(" + Re.ToString() + " ; " + Im.ToString() + ")";
}

public static class Mat2
{
    public static Cx[][] Zero2() => new[]
    {
        new[] { Cx.Zero, Cx.Zero },
        new[] { Cx.Zero, Cx.Zero }
    };

    public static bool Eq(Cx[][] a, Cx[][] b)
    {
        if (a.Length != b.Length) return false;
        for (int i = 0; i < a.Length; i++)
        {
            if (a[i].Length != b[i].Length) return false;
            for (int j = 0; j < a[i].Length; j++)
                if (a[i][j] != b[i][j]) return false;
        }
        return true;
    }

    public static Cx[][] Mul(Cx[][] a, Cx[][] b)
    {
        int n = a.Length;
        var res = new Cx[n][];
        for (int i = 0; i < n; i++)
        {
            res[i] = new Cx[n];
            for (int j = 0; j < n; j++)
            {
                var acc = Cx.Zero;
                for (int k = 0; k < n; k++) acc = acc + a[i][k] * b[k][j];
                res[i][j] = acc;
            }
        }
        return res;
    }

    public static Cx[][] Sub(Cx[][] a, Cx[][] b)
    {
        var res = new Cx[a.Length][];
        for (int i = 0; i < a.Length; i++)
        {
            res[i] = new Cx[a[i].Length];
            for (int j = 0; j < a[i].Length; j++) res[i][j] = a[i][j] - b[i][j];
        }
        return res;
    }
}

// Controle executable : leve si la valeur calculee differe de l'attendu.
public static class Verif
{
    public static void Check(bool condition, string label)
    {
        Console.WriteLine((condition ? "CONFORME" : "ÉCART") + " — " + label);
        if (!condition) throw new InvalidOperationException("Contrôle en échec : " + label);
    }
}

// Le groupe d'identification au bout qubit, CITE comme « thirty-two monomial
// matrices with fourth-root phases ». Nous l'enumerons : une matrice monomiale 2x2
// = une colonne par ligne, une phase (1, i, -1, -i) par colonne.
var racines = new[] { Cx.One, Cx.I, -Cx.One, -Cx.I };
var monomiales = new List<Cx[][]>();
foreach (int c0 in new[] { 0, 1 })
{
    foreach (var p1 in racines)
    {
        foreach (var p2 in racines)
        {
            var M = Mat2.Zero2();
            M[0][c0] = p1;
            M[1][1 - c0] = p2;
            monomiales.Add(M);
        }
    }
}
var scalaires = monomiales
    .Where(M => M[0][1] == Cx.Zero && M[1][0] == Cx.Zero && M[0][0] == M[1][1])
    .ToList();

Console.WriteLine("Matrices monomiales 2x2 a phases racines quatriemes : " + monomiales.Count
    + " -> " + (monomiales.Count == 32));
Console.WriteLine("Matrices scalaires parmi elles (le noyau cite) : " + scalaires.Count
    + " -> " + (scalaires.Count == 4));
Console.WriteLine("Arithmetique de Lagrange : 32 / 4 = " + new Rat(monomiales.Count, scalaires.Count)
    + " ; l'image sur les directions est d'ordre 8 (CITÉ)");

Verif.Check(racines.Length == 4 && racines.Distinct().Count() == 4,
    "quatre phases racines quatriemes distinctes");
Verif.Check(monomiales.Count == 32, "32 matrices monomiales 2x2");
Verif.Check(monomiales.All(M => M.Select(r => r.Count(x => x != Cx.Zero)).SequenceEqual(new[] { 1, 1 })),
    "chaque matrice monomiale a une seule entree non nulle par ligne");
Verif.Check(scalaires.Count == 4, "exactement 4 matrices scalaires (phases +-1, +-i sur l'identite)");
Verif.Check(new Rat(monomiales.Count, scalaires.Count) == Rat.FromInt(8), "32 / 4 = 8");


Matrices monomiales 2x2 a phases racines quatriemes : 32 -> True


Matrices scalaires parmi elles (le noyau cite) : 4 -> True


Arithmetique de Lagrange : 32 / 4 = 8 ; l'image sur les directions est d'ordre 8 (CITÉ)


CONFORME — quatre phases racines quatriemes distinctes


CONFORME — 32 matrices monomiales 2x2


CONFORME — chaque matrice monomiale a une seule entree non nulle par ligne


CONFORME — exactement 4 matrices scalaires (phases +-1, +-i sur l'identite)


CONFORME — 32 / 4 = 8


### Lecture du résultat (§53, comptes)

**Mesuré** : les comptes du groupe fini — 32 matrices monomiales à phases racines quatrièmes (2 choix de colonne × 4² phases), dont exactement 4 scalaires (les phases ±1, ±i sur l'identité). **Cité** : l'application de ce groupe aux trois directions dérivées — sa définition appartient à la construction — dont l'image est le groupe diédral d'ordre huit, de noyau les quatre phases scalaires. L'arithmétique 32 = 4 × 8 est la cohérence de Lagrange entre les deux comptes ; elle ne prouve pas l'image, elle la contraint.

## Les vingt-quatre rotations du réseau

L'image diédrale vit « a subgroup of index three in the twenty-four lattice rotations ». Les vingt-quatre rotations du réseau sont les matrices de permutation signée 3×3 de déterminant +1 — le groupe des rotations du cube, dans la langue de la littérature. Nous les énumérons, puis nous réalisons le sous-groupe d'ordre huit comme le **stabilisateur d'un axe** : les rotations du cube qui préservent un axe à signe près. Cette réalisation est **nôtre** — la source n'écrit pas la matrice — mais elle porte exactement les étiquettes citées : les directions y portent le signe ⊕ le naturel.

In [2]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Les 24 rotations du reseau : permutations signees 3x3 de determinant +1.
public static class Mat3
{
    public static int Det3(int[][] m)
        => m[0][0] * (m[1][1] * m[2][2] - m[1][2] * m[2][1])
         - m[0][1] * (m[1][0] * m[2][2] - m[1][2] * m[2][0])
         + m[0][2] * (m[1][0] * m[2][1] - m[1][1] * m[2][0]);

    public static int[][] Mul3(int[][] a, int[][] b)
    {
        var res = new int[3][];
        for (int i = 0; i < 3; i++)
        {
            res[i] = new int[3];
            for (int j = 0; j < 3; j++)
            {
                int acc = 0;
                for (int k = 0; k < 3; k++) acc += a[i][k] * b[k][j];
                res[i][j] = acc;
            }
        }
        return res;
    }

    public static bool Eq3(int[][] a, int[][] b)
    {
        for (int i = 0; i < a.Length; i++)
            for (int j = 0; j < a[i].Length; j++)
                if (a[i][j] != b[i][j]) return false;
        return true;
    }

    public static int[][] Identity3() => new[]
    {
        new[] { 1, 0, 0 },
        new[] { 0, 1, 0 },
        new[] { 0, 0, 1 }
    };

    public static int Ordre3(int[][] m)
    {
        var i3 = Identity3();
        int[][] r = m;
        int k = 1;
        while (!Eq3(r, i3)) { r = Mul3(r, m); k++; }
        return k;
    }
}

// Chaque element est une permutation 3x3, donc le conteneur est int[][].
int[][] perms3 =
{
    new[] { 0, 1, 2 }, new[] { 0, 2, 1 }, new[] { 1, 0, 2 },
    new[] { 1, 2, 0 }, new[] { 2, 0, 1 }, new[] { 2, 1, 0 }
};
int[][] signes3 =
{
    new[] { 1, 1, 1 }, new[] { 1, 1, -1 }, new[] { 1, -1, 1 }, new[] { 1, -1, -1 },
    new[] { -1, 1, 1 }, new[] { -1, 1, -1 }, new[] { -1, -1, 1 }, new[] { -1, -1, -1 }
};

var rotations24 = new List<int[][]>();
foreach (var perm in perms3)
{
    foreach (var sig in signes3)
    {
        var M = new int[3][];
        for (int i = 0; i < 3; i++) M[i] = new int[3];
        for (int i = 0; i < 3; i++) M[i][perm[i]] = sig[i];
        if (Mat3.Det3(M) == 1) rotations24.Add(M);
    }
}
Console.WriteLine("Permutations signees de determinant +1 : " + rotations24.Count
    + " -> " + (rotations24.Count == 24));

// Notre realisation du sous-groupe d'ordre 8 : le stabilisateur de l'axe z.
var axe = rotations24
    .Where(M => M[0][2] == 0 && M[1][2] == 0 && M[2][0] == 0 && M[2][1] == 0)
    .ToList();
var profil = axe.Select(Mat3.Ordre3).OrderBy(x => x).ToArray();
bool ferme = true;
foreach (var A in axe)
{
    foreach (var B in axe)
    {
        var produit = Mat3.Mul3(A, B);
        if (!axe.Any(g => Mat3.Eq3(g, produit))) ferme = false;
    }
}
Console.WriteLine("Stabilisateur de l'axe (ordre) : " + axe.Count
    + " -> " + (axe.Count == 8) + " ; stable par produit : " + ferme);
Console.WriteLine("Profil par ordres : [" + string.Join(", ", profil)
    + "] (D8 : cinq elements d'ordre 2, deux d'ordre 4 ; Q8 en aurait six d'ordre 4)");
Console.WriteLine("Indice dans les 24 : 24 / 8 = " + new Rat(24, axe.Count) + " (indice trois, CITÉ)");

Verif.Check(rotations24.Count == 24, "24 permutations signees de determinant +1");
Verif.Check(rotations24.All(M => Mat3.Det3(M) == 1), "chaque element de la liste a un determinant +1");
Verif.Check(axe.Count == 8, "stabilisateur d'axe d'ordre 8");
Verif.Check(ferme, "stabilisateur d'axe ferme par produit");
Verif.Check(profil.SequenceEqual(new[] { 1, 2, 2, 2, 2, 2, 4, 4 }),
    "profil par ordres [1, 2, 2, 2, 2, 2, 4, 4] (signature D8, pas Q8)");
Verif.Check(new Rat(24, axe.Count) == Rat.FromInt(3), "indice 24 / 8 = 3");


Permutations signees de determinant +1 : 24 -> True


Stabilisateur de l'axe (ordre) : 8 -> True ; stable par produit : True


Profil par ordres : [1, 2, 2, 2, 2, 2, 4, 4] (D8 : cinq elements d'ordre 2, deux d'ordre 4 ; Q8 en aurait six d'ordre 4)


Indice dans les 24 : 24 / 8 = 3 (indice trois, CITÉ)


CONFORME — 24 permutations signees de determinant +1


CONFORME — chaque element de la liste a un determinant +1


CONFORME — stabilisateur d'axe d'ordre 8


CONFORME — stabilisateur d'axe ferme par produit


CONFORME — profil par ordres [1, 2, 2, 2, 2, 2, 4, 4] (signature D8, pas Q8)


CONFORME — indice 24 / 8 = 3


In [3]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;

// La signature D8 se lit aussi sur les classes de conjugaison du stabilisateur :
// l'element neutre, le demi-tour central (trace -2 au bloc xy), les deux quarts
// de tour (ordre 4), et les QUATRE reflexions -- diagonales et hors-diagonales,
// deux classes distinctes de deux. Cinq classes = cinq irreductibles, les
// etiquettes A1, A2, B1, B2, E de la litterature [citees].
var classes = new Dictionary<(int ordre, int trace, bool horsDiag), int>();
foreach (var g in axe)
{
    int trace = g[0][0] + g[1][1];
    bool horsDiag = g[0][1] != 0;
    var cle = (Mat3.Ordre3(g), trace, horsDiag);
    classes.TryGetValue(cle, out int n);
    classes[cle] = n + 1;
}
// Un tuple n'implemente pas IComparable : on ordonne explicitement ses composantes.
foreach (var kv in classes
    .OrderBy(k => k.Key.ordre)
    .ThenBy(k => k.Key.trace)
    .ThenBy(k => k.Key.horsDiag))
{
    Console.WriteLine("ordre " + kv.Key.ordre
        + " | trace bloc xy " + kv.Key.trace
        + " | " + (kv.Key.horsDiag ? "hors-diagonale" : "diagonale")
        + " -> " + kv.Value + " element(s)");
}
Console.WriteLine("Classes distinguees : " + classes.Count
    + " -> " + (classes.Count == 5) + " (une par irreductible : A1, A2, B1, B2, E, citees)");

Verif.Check(classes.Count == 5, "cinq classes distinguees (une par irreductible : A1, A2, B1, B2, E)");
Verif.Check(classes.Values.Sum() == axe.Count, "les classes partitionnent les 8 elements du stabilisateur");
Verif.Check(classes.Values.OrderBy(v => v).SequenceEqual(new[] { 1, 1, 2, 2, 2 }),
    "tailles des classes {1, 1, 2, 2, 2} : neutre, demi-tour central, deux quarts de tour, deux reflexions diagonales, deux reflexions hors-diagonales");
Verif.Check(classes.Keys.Any(k => k.ordre == 1 && k.trace == 2 && !k.horsDiag),
    "classe de l'element neutre : ordre 1, trace du bloc xy 2, diagonale");
Verif.Check(classes.Keys.Any(k => k.ordre == 2 && k.trace == -2 && !k.horsDiag),
    "classe du demi-tour central : ordre 2, trace du bloc xy -2");
Verif.Check(classes.Keys.Count(k => k.ordre == 4) == 1 && classes[classes.Keys.First(k => k.ordre == 4)] == 2,
    "une seule classe d'ordre 4, de deux elements (les deux quarts de tour)");
Verif.Check(classes.Keys.Count(k => k.ordre == 2 && k.trace == 0) == 2,
    "deux classes d'ordre 2 et de trace nulle : reflexions diagonales et hors-diagonales");


ordre 1 | trace bloc xy 2 | diagonale -> 1 element(s)


ordre 2 | trace bloc xy -2 | diagonale -> 1 element(s)


ordre 2 | trace bloc xy 0 | diagonale -> 2 element(s)


ordre 2 | trace bloc xy 0 | hors-diagonale -> 2 element(s)


ordre 4 | trace bloc xy 0 | hors-diagonale -> 2 element(s)


Classes distinguees : 5 -> True (une par irreductible : A1, A2, B1, B2, E, citees)


... (3 outputs left out)

CONFORME — classe de l'element neutre : ordre 1, trace du bloc xy 2, diagonale


CONFORME — classe du demi-tour central : ordre 2, trace du bloc xy -2


CONFORME — une seule classe d'ordre 4, de deux elements (les deux quarts de tour)


CONFORME — deux classes d'ordre 2 et de trace nulle : reflexions diagonales et hors-diagonales


### Lecture du résultat (réseau)

**Mesuré** : 24 permutations signées de déterminant +1 ; le stabilisateur d'axe y forme un groupe d'ordre 8, fermé par produit, au profil d'ordres [1, 2, 2, 2, 2, 2, 4, 4] — la signature du diédral D8, que le groupe des quaternions Q8 n'a pas (six éléments d'ordre 4) — et à cinq classes de conjugaison. **Cité** : l'indice trois de l'image dans les vingt-quatre, et l'appartenance de cette réalisation à la construction de la source (la nôtre en est une incarnation, pas la définition).

## L'algèbre de Lie de dimension trois

« a Lie algebra of dimension three spanned by the ad matrices of the three derived directions with the brackets of the alternating symbol » — l'algèbre des rotations est engendrée par les trois directions dérivées, aux crochets $[T_a, T_b] = \epsilon_{abc} T_c$. Le compendium imprime le triple transposé au chapitre suivant (p. 147, dérivation du Thm 55.1) : $\left(\begin{smallmatrix}0 & -i/2\\ -i/2 & 0\end{smallmatrix}\right)$, $\left(\begin{smallmatrix}0 & 1/2\\ -1/2 & 0\end{smallmatrix}\right)$, $\left(\begin{smallmatrix}-i/2 & 0\\ 0 & i/2\end{smallmatrix}\right)$ — à un facteur près, les matrices de Pauli. Nous en mesurons les constantes de structure exactes dans $\mathbb{Q}(i)$, puis la ré-orientation d'un signe qui les ramène au symbole alterné — le compendium dit lui-même, au ch. 55, que la base littérale portée demande une ré-orientation pour retrouver la table des crochets.

In [4]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Algèbre matricielle sur Q(i) exact : commutateur, négation, proportionnalité.
public static class Lie
{
    public static Cx[][] Crochet(Cx[][] a, Cx[][] b) => Mat2.Sub(Mat2.Mul(a, b), Mat2.Mul(b, a));

    public static Cx[][] Neg(Cx[][] m)
    {
        var r = new Cx[m.Length][];
        for (int i = 0; i < m.Length; i++)
        {
            r[i] = new Cx[m[i].Length];
            for (int j = 0; j < m[i].Length; j++) r[i][j] = -m[i][j];
        }
        return r;
    }

    public static Cx[][] Scale(Cx[][] m, Rat k)
    {
        var r = new Cx[m.Length][];
        for (int i = 0; i < m.Length; i++)
        {
            r[i] = new Cx[m[i].Length];
            for (int j = 0; j < m[i].Length; j++) r[i][j] = Cx.Scale(m[i][j], k);
        }
        return r;
    }

    // Division complexe exacte : a / b, rationnels reduits, aucun flottant.
    private static Cx Div(Cx a, Cx b)
    {
        var den = b.Re * b.Re + b.Im * b.Im;
        return new Cx((a.Re * b.Re + a.Im * b.Im) / den, (a.Im * b.Re - a.Re * b.Im) / den);
    }

    // Cherche g et le scalaire complexe c tels que x = c*g : on prend le premier
    // coefficient non nul de g pour fixer c, puis on verifie l'egalite entiere.
    public static (int k, Cx c) Proportionnel(Cx[][] x, Cx[][][] baseTriple)
    {
        for (int k = 0; k < baseTriple.Length; k++)
        {
            var g = baseTriple[k];
            Cx? c = null;
            for (int i = 0; i < 2 && c is null; i++)
            {
                for (int j = 0; j < 2 && c is null; j++)
                {
                    if (g[i][j] != Cx.Zero) c = Div(x[i][j], g[i][j]);
                }
            }
            if (c is null) continue;
            bool ok = true;
            for (int i = 0; i < 2 && ok; i++)
            {
                for (int j = 0; j < 2 && ok; j++)
                {
                    if (c * g[i][j] != x[i][j]) ok = false;
                }
            }
            if (ok) return (k, c);
        }
        throw new InvalidOperationException("Aucune proportionnalite trouvee.");
    }
}

// Le triple littéral du compendium (ch. 55, p. 147, cité).
var DEMI = new Cx(new Rat(1, 2), Rat.Zero);
var G1 = new[]
{
    new[] { Cx.Zero, Cx.Scale(-Cx.I, new Rat(1, 2)) },
    new[] { Cx.Scale(-Cx.I, new Rat(1, 2)), Cx.Zero }
};
var G2 = new[]
{
    new[] { Cx.Zero, DEMI },
    new[] { -DEMI, Cx.Zero }
};
var G3 = new[]
{
    new[] { Cx.Scale(-Cx.I, new Rat(1, 2)), Cx.Zero },
    new[] { Cx.Zero, Cx.Scale(Cx.I, new Rat(1, 2)) }
};
var litteral = new[] { G1, G2, G3 };

Console.WriteLine("Constantes de structure du triple littéral :");
var constantes = new List<Rat>();
foreach (var ab in new[] { (0, 1), (1, 2), (2, 0) })
{
    var a = ab.Item1;
    var b = ab.Item2;
    var prop = Lie.Proportionnel(Lie.Crochet(litteral[a], litteral[b]), litteral);
    constantes.Add(prop.c.Re);
    Console.WriteLine("  [G" + (a + 1).ToString(CultureInfo.InvariantCulture)
        + ",G" + (b + 1).ToString(CultureInfo.InvariantCulture) + "] = "
        + prop.c.Re.ToString() + " * G" + (prop.k + 1).ToString(CultureInfo.InvariantCulture));
}
bool litteral_ok = constantes.All(c => c == Rat.FromInt(-1));
Console.WriteLine("Les trois constantes au signe moins : " + litteral_ok);

// Re-orientation : un seul signe retourne (le deuxieme generateur) et les crochets
// deviennent exactement ceux du symbole alterne.
var T = new[] { litteral[0], Lie.Neg(litteral[1]), litteral[2] };
var attendu = new Dictionary<(int, int), int> { { (0, 1), 2 }, { (1, 2), 0 }, { (2, 0), 1 } };
bool epsilon_exact = attendu.All(ent => Mat2.Eq(Lie.Crochet(T[ent.Key.Item1], T[ent.Key.Item2]), T[ent.Value]));
Console.WriteLine("Re-orienté (flip de G2) : [Ta,Tb] = eps_abl*Tc exact -> " + epsilon_exact);
var moins_quart_I = new[]
{
    new[] { new Cx(new Rat(-1, 4), Rat.Zero), Cx.Zero },
    new[] { Cx.Zero, new Cx(new Rat(-1, 4), Rat.Zero) }
};
var carres = T.Select(t => Mat2.Eq(Mat2.Mul(t, t), moins_quart_I)).ToArray();
Console.WriteLine("Tj^2 = -(1/4)*I pour j = 1, 2, 3 : "
    + string.Join(", ", carres.Select(b => b ? "vrai" : "faux"))
    + " -> " + carres.All(b => b));

Verif.Check(constantes.Count == 3 && litteral_ok,
    "constantes de structure du triple litteral : (-1, -1, -1)");
Verif.Check(epsilon_exact, "re-orientation d'un seul signe : [Ta,Tb] = eps_abl*Tc exact");
Verif.Check(carres.Length == 3 && carres.All(b => b), "Tj^2 = -(1/4)*I pour les trois axes");
Verif.Check(T.Select(t => Mat2.Mul(t, t)).Select(c => c[0][0].Re).All(v => v == new Rat(-1, 4)),
    "coefficient diagonal mesure de Tj^2 : -1/4");


Constantes de structure du triple littéral :


  [G1,G2] = -1 * G3


  [G2,G3] = -1 * G1


  [G3,G1] = -1 * G2


Les trois constantes au signe moins : True


Re-orienté (flip de G2) : [Ta,Tb] = eps_abl*Tc exact -> True


Tj^2 = -(1/4)*I pour j = 1, 2, 3 : vrai, vrai, vrai -> True


CONFORME — constantes de structure du triple litteral : (-1, -1, -1)


CONFORME — re-orientation d'un seul signe : [Ta,Tb] = eps_abl*Tc exact


CONFORME — Tj^2 = -(1/4)*I pour les trois axes


CONFORME — coefficient diagonal mesure de Tj^2 : -1/4


### Les exponentielles aux points rationnels et le double recouvrement

Puisque $T_j^2 = -\tfrac{1}{4} I$, l'exponentielle a la forme fermée $\exp(\theta T_j) = \cos(\theta/2) I + 2\sin(\theta/2) T_j$. Aux angles rationnels en demi-tours, tout est exact dans $\mathbb{Q}(i)$ : le demi-tour $\theta = \pi$ donne $2T_3 = -i\sigma_z$ ; le tour complet $\theta = 2\pi$ donne $-I$ **au bout qubit** — pendant que l'action conjuguée sur les directions est l'identité. C'est le double recouvrement de la littérature, mesuré : « the full rotation is minus one at every qubit end and plus one on the directions ».

In [5]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Demi-tour : exp(pi*T3) = cos(pi/2) I + 2 sin(pi/2) T3 = 2*T3 = -i*sigma_z.
var demi_tour = Lie.Scale(T[2], new Rat(2, 1));
var attendu_demi = new[]
{
    new[] { new Cx(Rat.Zero, new Rat(-1, 1)), Cx.Zero },
    new[] { Cx.Zero, new Cx(Rat.Zero, Rat.One) }
};
bool demi_ok = Mat2.Eq(demi_tour, attendu_demi);
Console.WriteLine("exp(pi*T3) = 2*T3 = diag(-i, i) = -i*sigma_z -> " + demi_ok);

// Tour complet : forme fermee cos(pi) = -1, sin(pi) = 0 -> -I. Et comme
// exp(2*pi*Tj) = (exp(pi*Tj))^2 = (2*Tj)^2 = 4*Tj^2 avec Tj^2 = -(1/4)I mesure
// ci-dessus, le CARRE du demi-tour est la mesure exacte du tour complet.
var moins_I = new[]
{
    new[] { -Cx.One, Cx.Zero },
    new[] { Cx.Zero, -Cx.One }
};
var exp2pi = T.Select(t => Mat2.Eq(Mat2.Mul(Lie.Scale(t, new Rat(2, 1)), Lie.Scale(t, new Rat(2, 1))), moins_I)).ToArray();
Console.WriteLine("exp(2*pi*Tj) = (2*Tj)^2 = 4*Tj^2 = -I au bout qubit, chaque axe -> "
    + string.Join(", ", exp2pi.Select(b => b ? "vrai" : "faux")) + " -> " + exp2pi.All(b => b));

// L'action adjointe sur les directions : ad_T3 en base re-orientee. La mesure
// porte sur la GENERATRICE T3 (les crochets [T3, T1] = T2, [T3, T2] = -T1,
// [T3, T3] = 0). L'exponentielle Ad(exp(pi/2 T3)) = exp(pi/2 * ad_T3) est
// ensuite CALCULEE a partir de cette matrice, pas ecrite a la main.
int[] CoeffAd(int k, int j)
{
    var c = Lie.Crochet(T[k], T[j]);
    var col = new int[3];
    for (int i = 0; i < 3; i++)
    {
        if (Mat2.Eq(c, T[i])) col[i] = 1;
        else if (Mat2.Eq(c, Lie.Neg(T[i]))) col[i] = -1;
    }
    return col;
}
var cols = Enumerable.Range(0, 3).Select(j => CoeffAd(2, j)).ToArray();
var ad_T3 = new int[3][];
for (int i = 0; i < 3; i++) ad_T3[i] = new int[3];
for (int i = 0; i < 3; i++)
    for (int j = 0; j < 3; j++) ad_T3[i][j] = cols[j][i];
Console.WriteLine("ad_T3 (base re-orientée) = [[" + string.Join(", ", ad_T3[0]) + "], ["
    + string.Join(", ", ad_T3[1]) + "], [" + string.Join(", ", ad_T3[2]) + "]]");
bool ad_attendu = ad_T3[0][0] == 0 && ad_T3[0][1] == -1 && ad_T3[0][2] == 0
    && ad_T3[1][0] == 1 && ad_T3[1][1] == 0 && ad_T3[1][2] == 0
    && ad_T3[2][0] == 0 && ad_T3[2][1] == 0 && ad_T3[2][2] == 0;
Console.WriteLine("ad de la génératrice T3 = J : T1 -> T2, T2 -> -T1, partie planaire du quart de tour -> " + ad_attendu);

// ad_T3 a la structure J : J^3 = -J au bloc planaire, nullite sur l'axe. Donc
// exp(theta J) = I + sin(theta) J + (1 - cos(theta)) J^2 ; au quart de tour
// sin = 1 et cos = 0, d'ou exp = I + J + J^2 -- exact, sans flottants.
var J2 = Mat3.Mul3(ad_T3, ad_T3);
var J3 = Mat3.Mul3(J2, ad_T3);
bool J2_ok = J2[0][0] == -1 && J2[0][1] == 0 && J2[0][2] == 0
    && J2[1][0] == 0 && J2[1][1] == -1 && J2[1][2] == 0
    && J2[2][0] == 0 && J2[2][1] == 0 && J2[2][2] == 0;
bool J3_ok = Mat3.Eq3(J3, ad_T3.Select(l => l.Select(v => -v).ToArray()).ToArray());
Console.WriteLine("ad_T3^2 = diag(-1, -1, 0) -> " + J2_ok + "; ad_T3^3 = -ad_T3 -> " + J3_ok);

var I3 = Mat3.Identity3();
var quart = new int[3][];
for (int i = 0; i < 3; i++)
{
    quart[i] = new int[3];
    for (int j = 0; j < 3; j++) quart[i][j] = I3[i][j] + ad_T3[i][j] + J2[i][j];
}
Console.WriteLine("exp(pi/2 * ad_T3) = I + J + J^2 = [[" + string.Join(", ", quart[0]) + "], ["
    + string.Join(", ", quart[1]) + "], [" + string.Join(", ", quart[2]) + "]]");
bool quart_ok = quart[0][0] == 0 && quart[0][1] == -1 && quart[0][2] == 0
    && quart[1][0] == 1 && quart[1][1] == 0 && quart[1][2] == 0
    && quart[2][0] == 0 && quart[2][1] == 0 && quart[2][2] == 1;
Console.WriteLine("Le quart de tour calcule est le plan de rotation applique -> " + quart_ok);
var q2 = Mat3.Mul3(quart, quart);
var q4 = Mat3.Mul3(q2, q2);
bool ad_tour_complet = Mat3.Eq3(q4, I3);
Console.WriteLine("Le quart de tour sur les directions, à la puissance 4 : identité (quart^4 = I3) -> " + ad_tour_complet);
bool double_couvre = demi_ok && exp2pi.All(b => b) && ad_tour_complet;
Console.WriteLine("Double recouvrement : -I au bout qubit, +I sur les directions -> " + double_couvre);

Verif.Check(demi_ok, "exp(pi*T3) = 2*T3 = diag(-i, i) = -i*sigma_z");
Verif.Check(exp2pi.Length == 3 && exp2pi.All(b => b), "exp(2*pi*Tj) = -I au bout qubit pour les trois axes");
Verif.Check(ad_attendu, "ad_T3 = J : T1 -> T2, T2 -> -T1");
Verif.Check(J2_ok && J3_ok, "J^2 = diag(-1,-1,0) et J^3 = -J");
Verif.Check(quart_ok, "exp(pi/2 * ad_T3) = I + J + J^2 = rotation planaire");
Verif.Check(ad_tour_complet, "quart^4 = I3 sur les directions");
Verif.Check(double_couvre, "double recouvrement : -I au bout qubit, +I sur les directions");


exp(pi*T3) = 2*T3 = diag(-i, i) = -i*sigma_z -> True


exp(2*pi*Tj) = (2*Tj)^2 = 4*Tj^2 = -I au bout qubit, chaque axe -> vrai, vrai, vrai -> True


ad_T3 (base re-orientée) = [[0, -1, 0], [1, 0, 0], [0, 0, 0]]


ad de la génératrice T3 = J : T1 -> T2, T2 -> -T1, partie planaire du quart de tour -> True


ad_T3^2 = diag(-1, -1, 0) -> True; ad_T3^3 = -ad_T3 -> True


exp(pi/2 * ad_T3) = I + J + J^2 = [[0, -1, 0], [1, 0, 0], [0, 0, 1]]


... (6 outputs left out)

CONFORME — J^2 = diag(-1,-1,0) et J^3 = -J


CONFORME — exp(pi/2 * ad_T3) = I + J + J^2 = rotation planaire


CONFORME — quart^4 = I3 sur les directions


CONFORME — double recouvrement : -I au bout qubit, +I sur les directions


### Lecture du résultat (algèbre)

**Mesuré** : les constantes de structure du triple imprimé — $[G_1,G_2] = -G_3$, $[G_2,G_3] = -G_1$, $[G_3,G_1] = -G_2$, toutes les trois au signe moins — et la ré-orientation d'un seul signe qui les amène au symbole alterné $\epsilon_{abc}$ exact ; $T_j^2 = -\tfrac14 I$ ; le demi-tour $-i\sigma_z$ ; le tour complet $-I$ au bout qubit contre $+I$ sur les directions. **Cité** : l'identification de cette algèbre à celle du groupe des rotations de la construction (le stabilisateur de l'unité hermitienne dans les isométries de la scission de la trace), le relevé par conjugaison, et les treize corps typés à chaque bout qubit.

## Les étiquettes de caractères et le commutant

« the directions carry $A_2 \oplus E$ and the pair space of the end $A_1 \oplus A_2 \oplus E$, so its commutant has dimension three. » Les étiquettes sont celles de la littérature (les cinq irreductibles du diédral). Sur **notre** réalisation — le stabilisateur d'axe agissant sur $\mathbb{R}^3$ — nous mesurons les caractères, la décomposition et les commutants : le caractère des directions est exactement le signe de l'axe plus la trace du bloc naturel, et le commutant de l'espace de paire (trivial ⊕ directions) vaut trois, la valeur imprimée.

In [6]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Elimination lineaire exacte sur les rationnels : dimension du commutant.
public static class Lin
{
    public static int Commutant(List<int[][]> mats, int n)
    {
        var lignes = new List<Rat[]>();
        foreach (var g in mats)
        {
            for (int i = 0; i < n; i++)
            {
                for (int j = 0; j < n; j++)
                {
                    var ligne = new Rat[n * n];
                    for (int k = 0; k < n * n; k++) ligne[k] = Rat.Zero;
                    for (int k = 0; k < n; k++)
                    {
                        ligne[n * i + k] = ligne[n * i + k] + Rat.FromInt(g[k][j]);
                        ligne[n * k + j] = ligne[n * k + j] - Rat.FromInt(g[i][k]);
                    }
                    lignes.Add(ligne);
                }
            }
        }
        int piv = 0;
        for (int col = 0; col < n * n; col++)
        {
            int sel = -1;
            for (int r = piv; r < lignes.Count; r++)
            {
                if (lignes[r][col] != Rat.Zero) { sel = r; break; }
            }
            if (sel < 0) continue;
            var tmp = lignes[piv];
            lignes[piv] = lignes[sel];
            lignes[sel] = tmp;
            var pv = lignes[piv][col];
            for (int k = 0; k < n * n; k++) lignes[piv][k] = lignes[piv][k] / pv;
            for (int r = 0; r < lignes.Count; r++)
            {
                if (r == piv || lignes[r][col] == Rat.Zero) continue;
                var f = lignes[r][col];
                for (int k = 0; k < n * n; k++)
                    lignes[r][k] = lignes[r][k] - f * lignes[piv][k];
            }
            piv++;
        }
        return n * n - piv;
    }
}

// Caracteres sur notre realisation (etiquettes citees, decomposition mesuree).
Rat ChiA1(int[][] g) => Rat.One;
Rat ChiA2(int[][] g) => Rat.FromInt(g[2][2]);
Rat ChiE(int[][] g) => Rat.FromInt(g[0][0] + g[1][1]);
Rat ChiDir(int[][] g) => Rat.FromInt(g[0][0] + g[1][1] + g[2][2]);
Rat ChiPair(int[][] g) => Rat.One + ChiDir(g);

Rat ScalaireInterne(Func<int[][], Rat> f, Func<int[][], Rat> h)
{
    var s = Rat.Zero;
    foreach (var g in axe) s = s + f(g) * h(g);
    return s / new Rat(axe.Count, 1);
}

var s_dir_A1 = ScalaireInterne(ChiDir, ChiA1);
var s_dir_A2 = ScalaireInterne(ChiDir, ChiA2);
var s_dir_E = ScalaireInterne(ChiDir, ChiE);
var s_pair_A1 = ScalaireInterne(ChiPair, ChiA1);
var s_pair_A2 = ScalaireInterne(ChiPair, ChiA2);
var s_pair_E = ScalaireInterne(ChiPair, ChiE);
Console.WriteLine("<chi_dir, A1> = " + s_dir_A1.ToString() + " (aucune composante triviale)");
Console.WriteLine("<chi_dir, A2> = " + s_dir_A2.ToString() + "; <chi_dir, E> = " + s_dir_E.ToString());
Console.WriteLine("<chi_pair, A1> = " + s_pair_A1.ToString() + "; <chi_pair, A2> = "
    + s_pair_A2.ToString() + "; <chi_pair, E> = " + s_pair_E.ToString());

// Commutants : dimension de l'espace des matrices qui commutent a tout le groupe.
var pair4 = new List<int[][]>();
foreach (var g in axe)
{
    var B = new int[4][];
    for (int i = 0; i < 4; i++) B[i] = new int[4];
    B[0][0] = 1;
    for (int i = 0; i < 3; i++)
        for (int j = 0; j < 3; j++)
            B[1 + i][1 + j] = g[i][j];
    pair4.Add(B);
}
int d_dir = Lin.Commutant(axe, 3);
int d_pair = Lin.Commutant(pair4, 4);
Console.WriteLine("Commutant du modele directions (A2+E) : " + d_dir.ToString(CultureInfo.InvariantCulture)
    + " ; du modele espace de paire (A1+A2+E) : " + d_pair.ToString(CultureInfo.InvariantCulture)
    + " -> la valeur trois imprimee est celle de l'espace de paire sur ce modele");

Verif.Check(s_dir_A1 == Rat.Zero, "<chi_dir, A1> = 0 (aucune composante triviale)");
Verif.Check(s_dir_A2 == Rat.One && s_dir_E == Rat.One, "<chi_dir, A2> = 1 et <chi_dir, E> = 1");
Verif.Check(s_pair_A1 == Rat.One && s_pair_A2 == Rat.One && s_pair_E == Rat.One,
    "<chi_pair, A1> = <chi_pair, A2> = <chi_pair, E> = 1");
Verif.Check(d_dir == 2, "commutant du modele directions : dimension 2");
Verif.Check(d_pair == 3, "commutant de l'espace de paire : dimension 3 (valeur imprimee)");


<chi_dir, A1> = 0 (aucune composante triviale)


<chi_dir, A2> = 1; <chi_dir, E> = 1


<chi_pair, A1> = 1; <chi_pair, A2> = 1; <chi_pair, E> = 1


Commutant du modele directions (A2+E) : 2 ; du modele espace de paire (A1+A2+E) : 3 -> la valeur trois imprimee est celle de l'espace de paire sur ce modele


CONFORME — <chi_dir, A1> = 0 (aucune composante triviale)


CONFORME — <chi_dir, A2> = 1 et <chi_dir, E> = 1


CONFORME — <chi_pair, A1> = <chi_pair, A2> = <chi_pair, E> = 1


CONFORME — commutant du modele directions : dimension 2


CONFORME — commutant de l'espace de paire : dimension 3 (valeur imprimee)


### Lecture du résultat (caractères)

**Mesuré** : sur notre réalisation, le caractère des directions se décompose en $A_2 \oplus E$ (produits scalaires 0, 1, 1) et celui de l'espace de paire en $A_1 \oplus A_2 \oplus E$ (1, 1, 1) ; le commutant de l'espace de paire vaut **trois**, celui des directions deux. La phrase imprimée — « so its commutant has dimension three » — suit immédiatement les deux étiquettes ; la valeur trois est, sur notre modèle, celle de **l'espace de paire** ($1 + 1 + 1$), ce qui est aussi la lecture grammaticale. **Cité** : les étiquettes elles-mêmes, attachées à la construction de la source.

## Ce que le census porte (cité)

Le census du chapitre enumere les 102 bouts qubit des corps de la construction — sites, liens et types de paire — et verifie le stabilisateur a chacun. Nous citons sa table et ses comptes : ils ne se rejouent pas ici (la liste des 102 objets n'est pas imprimee dans l'extrait).

In [7]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;

// Les comptes du census, cites tel qu'imprimes (extrait p. 143).
// DONNEES CITEES : ces nombres sont saisis comme cites, ils ne sont pas recalcules ici.
var census = new Dictionary<string, int>
{
    ["bouts qubit enumeres"] = 102,
    ["corps de la construction types"] = 13,
    ["controles lus pareil aux deux moteurs"] = 13,
    ["contre-tests aux lettres epinglees"] = 5,
    ["longueur de la plus longue chaine de premices"] = 12,
};
foreach (var kv in census)
{
    Console.WriteLine(kv.Key + " : " + kv.Value.ToString(CultureInfo.InvariantCulture) + " (CITÉ)");
}
Console.WriteLine("Contre-test nomme : uniterre frappee -> plus de groupe, une algebra d'isometrie de dimension trois (cite)");
Console.WriteLine("Second moteur construit a part : chaque cellule reproduite egale (cite)");
Console.WriteLine("Rappel : donnees citees, non rejouees dans ce carnet (la table des 102 bouts n'est pas imprimee dans l'extrait).");


bouts qubit enumeres : 102 (CITÉ)


corps de la construction types : 13 (CITÉ)


controles lus pareil aux deux moteurs : 13 (CITÉ)


contre-tests aux lettres epinglees : 5 (CITÉ)


longueur de la plus longue chaine de premices : 12 (CITÉ)


Contre-test nomme : uniterre frappee -> plus de groupe, une algebra d'isometrie de dimension trois (cite)


Second moteur construit a part : chaque cellule reproduite egale (cite)


Rappel : donnees citees, non rejouees dans ce carnet (la table des 102 bouts n'est pas imprimee dans l'extrait).


## Carrefours avec la série

- **Chaînes-09** : l'éq. 76 mesurait $\tau = 1/4$, réciproque du rapport des carrés — les **quatre** cartes de signe au rapport carré. Ici les phases du groupe d'identification sont les **racines quatrièmes** de l'unité, et les quatre signes de l'éq. 76 sont le sous-groupe d'ordre quatre des phases réelles. La structure « quart » est la même des deux côtés, à des objets différents.
- **Chaînes-01** : le lien qutrit-qubit dont l'algèbre de cadre porte les directions dérivées — le même objet dont l'ordre coordinate lisait le centre unidimensionnel.
- **Chaînes-05** : les niveaux du ch. 44 étaient lus au corps électron du lien $(2,3)$ frappé ; les docks typés ici (propre au bout proche, impropre au bout loin) sont ceux de la même construction des corps.

## Ouvert — ligne à ligne

1. **Une direction par paire depuis le support partagé** : une affirmation d'unicité « where the construction admits several readings, decided at one of four and superseded by the automorphism theorem » (§53) — l'un des quatre choix est fait, la lecture automorphique le remplace, l'unicité reste un résultat ouvert.
2. **Le signe de site propre** : « equals the standing swap sign at seventeen of the eighteen objects », parting au tau — son site propre est un port qubit-qubit apparié à une feuille que le transport n'ouvre à aucune orientation. Le parting est nommé, non expliqué : ouvert.
3. **L'étiquette orbitale** conservée à chaque instance d'interaction — **conditionnelle** sur la clause à un acte (une prémisse de la série, pas un théorème établi).
4. **La rotation d'angle générique** est hors du groupe — le continu est fermé au premier résultat du chapitre : la construction ne livre que le groupe fini et son algèbre, pas SO(3) tout entier ; c'est une limite de l'objet, pas une lacune de preuve.
5. **Le census des 102 bouts** : cité, non rejoué — la table d'objets n'est pas imprimée dans l'extrait ; nos comptes mesurés portent les groupes finis et l'algèbre, pas l'attribution typée corps par corps.

### Exercice 1 — l'autre moitié du réseau

Les 48 permutations signées se partagent en 24 de déterminant +1 (les rotations, comptées ci-dessus) et 24 de déterminant −1. Énumérer les secondes et vérifier le partage.

In [8]:
#nullable enable
using System;
using System.Globalization;

// Exercice 1 — Etendre l'enumeration aux permutations signees de determinant -1.
// TODO etudiant : enumerer les permutations signees 3x3 de determinant -1
// et verifier 24 + 24 = 48 (les « improprement types » du census sont a lire dans ce miroir)
int impropres = 0; // valeur neutre, a calculer par l'etudiant
Console.WriteLine("Exercice a completer — impropres = "
    + impropres.ToString(CultureInfo.InvariantCulture)
    + " (valeur neutre ; TODO etudiant : enumerer les determinant -1 et verifier le partage 24 + 24 = 48)");


Exercice a completer — impropres = 0 (valeur neutre ; TODO etudiant : enumerer les determinant -1 et verifier le partage 24 + 24 = 48)


### Exercice 2 — un autre point rationnel

L'exponentielle exacte vit aux points rationnels du cercle. Au point pythagoricien $(\cos\theta, \sin\theta) = (5/13, 12/13)$, écrire la matrice de rotation de l'action adjointe dans le plan $(T_1, T_2)$ et vérifier exactement qu'elle préserve la forme euclidienne.

In [9]:
#nullable enable
using System;
using System.Globalization;

// Exercice 2 — un autre point rationnel.
var c = new Rat(5, 13);
var s = new Rat(12, 13);
int[][]? resultat = null; // valeur neutre : la matrice reste a construire par l'etudiant
Console.WriteLine("Point pythagoricien : cos = " + c.ToString() + ", sin = " + s.ToString());
Console.WriteLine("Exercice a completer — resultat = " + (resultat is null ? "null (valeur neutre)" : "matrice construite"));
// TODO etudiant : construire la matrice de rotation de l'action adjointe dans le plan (T1, T2)
// a partir de c et s, puis verifier exactement R^T R = I et det R = 1 en arithmetique rationnelle exacte.


Point pythagoricien : cos = 5/13, sin = 12/13


Exercice a completer — resultat = null (valeur neutre)


### Exercice 3 — le commutant du groupe entier

Le commutant du stabilisateur d'axe (ordre 8) sur les directions vaut deux, sur l'espace de paire trois. Que vaut le commutant du groupe **entier** des 24 rotations sur $\mathbb{R}^3$ ? Mesurer avec la même fonction `commutant`.

In [10]:
#nullable enable
using System;
using System.Globalization;

// Exercice 3 — le commutant du groupe entier.
int compte = 0; // valeur neutre, a mesurer par l'etudiant
Console.WriteLine("Exercice a completer — compte = "
    + compte.ToString(CultureInfo.InvariantCulture)
    + " (valeur neutre ; TODO etudiant : mesurer Lin.Commutant(rotations24, 3) et interpreter le resultat)");


Exercice a completer — compte = 0 (valeur neutre ; TODO etudiant : mesurer Lin.Commutant(rotations24, 3) et interpreter le resultat)


## Conclusion

Le groupe des rotations n'est pas posé, il est dérivé : trois directions, leurs matrices *ad*, une algèbre de dimension trois aux crochets du symbole alterné — et un double recouvrement mesuré ($-I$ au bout qubit, $+I$ sur les directions). L'image finie est le diédral d'ordre huit à indice trois dans les vingt-quatre rotations du réseau, et le census typé des 102 bouts reste cité, non rejoué. « The rotations fixed, the next chapter joins them to the order coordinate in the Lorentz group » — le chapitre suivant soude rotations et coordonnée d'ordre dans le groupe de Lorentz.

In [11]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Releve final : les checks du carnet.
var checks = new Dictionary<string, bool>
{
    ["32 monomiales, 4 scalaires, 32/4 = 8"] = monomiales.Count == 32 && scalaires.Count == 4
        && new Rat(monomiales.Count, scalaires.Count) == Rat.FromInt(8),
    ["24 rotations du reseau"] = rotations24.Count == 24,
    ["stabilisateur d'axe = 8, ferme, profil D8"] = axe.Count == 8 && ferme
        && profil.SequenceEqual(new[] { 1, 2, 2, 2, 2, 2, 4, 4 }),
    ["cinq classes (A1, A2, B1, B2, E)"] = classes.Count == 5,
    ["crochets litteraux = (-1, -1, -1)"] = litteral_ok,
    ["re-orientation -> eps_abl exact"] = epsilon_exact,
    ["Tj^2 = -(1/4)I"] = carres.All(b => b),
    ["exp(2*pi*Tj) = -I au bout, +I sur directions"] = double_couvre,
    ["chi_dir = A2 + E, chi_pair = A1 + A2 + E"] = s_dir_A2 == Rat.One
        && s_dir_E == Rat.One && s_pair_A1 == Rat.One,
    ["quart de tour = exp(pi/2 ad_T3), puissance 4 = I3"] = quart_ok && ad_tour_complet,
    ["commutant espace de paire = 3"] = d_pair == 3,
};
int ok = checks.Values.Count(v => v);
foreach (var kv in checks)
{
    Console.WriteLine(kv.Key + " -> " + (kv.Value ? "CONFORME" : "ÉCART"));
}
Console.WriteLine("Total : " + ok.ToString(CultureInfo.InvariantCulture) + " / "
    + checks.Count.ToString(CultureInfo.InvariantCulture));
Verif.Check(ok == checks.Count, "releve final : tous les checks du carnet sont conformes");


32 monomiales, 4 scalaires, 32/4 = 8 -> CONFORME


24 rotations du reseau -> CONFORME


stabilisateur d'axe = 8, ferme, profil D8 -> CONFORME


cinq classes (A1, A2, B1, B2, E) -> CONFORME


crochets litteraux = (-1, -1, -1) -> CONFORME


re-orientation -> eps_abl exact -> CONFORME


... (3 outputs left out)

quart de tour = exp(pi/2 ad_T3), puissance 4 = I3 -> CONFORME


commutant espace de paire = 3 -> CONFORME


Total : 11 / 11


CONFORME — releve final : tous les checks du carnet sont conformes


## À retenir

1. **Le groupe n'est pas posé, il est dérivé** : trois directions, leurs matrices *ad*, une
   algèbre de dimension trois fermée par les crochets du symbole alterné — et $T_j^2 = -\tfrac14 I$
   aux trois axes, mesuré.
2. **Les comptes se referment** : $32$ matrices monomiales à phases racines quatrièmes, exactement
   $4$ scalaires, image d'ordre $32/4 = 8$ par Lagrange ; et $24$ rotations du réseau
   (permutations signées de déterminant $+1$), dont le stabilisateur d'axe est un sous-groupe
   d'indice $3$, fermé par produit, profil par ordres $[1,2,2,2,2,2,4,4]$ — $D_8$, pas $Q_8$.
3. **Cinq classes pour cinq irréductibles** : le tri (ordre, trace du bloc $xy$, diagonale)
   distingue exactement cinq classes, et les caractères se décomposent : directions
   $= A_2 \oplus E$, espace de paire $= A_1 \oplus A_2 \oplus E$, commutants $2$ et $3$.
4. **Le quart de tour adjoint se calcule** : $\mathrm{ad}_{T_3} = J$ vérifie $J^3 = -J$, donc
   $\exp(\tfrac{\pi}{2} J) = I + J + J^2$ — la forme fermée donne exactement la rotation
   planaire, et sa puissance quatrième vaut $I_3$ **sur les directions**.
5. **Le double recouvrement est mesuré, pas cité** : $(2T_j)^2 = 4T_j^2 = -I$ au bout qubit pour
   les trois axes, pendant que le tour complet agit trivialement sur les directions — c'est
   l'écart $\pm I$ qui distingue les deux bouts.

## Vérifiez votre compréhension

1. Pourquoi $32/4 = 8$ ne prouve-t-il pas que l'image est diédrale, et que faut-il de plus pour
   l'établir ?
2. Comment le profil par ordres distingue-t-il $D_8$ de $Q_8$, et que lit-on sur ses cinq classes
   de conjugaison ?
3. Que signifie « le triple littéral à constantes de structure $(-1,-1,-1)$ » et qu'est-ce qu'une
   ré-orientation de signe y change ?
4. Pourquoi $\exp(\tfrac{\pi}{2}\,\mathrm{ad}_{T_3})$ vaut-il $I + J + J^2$, et que mesure
   la puissance quatrième de cette matrice ?

<details>
<summary>Réponses</summary>

1. L'arithmétique $32 = 4 \times 8$ est une **cohérence** de Lagrange : elle contraint l'indice du
   noyau, elle ne dit rien de la structure de l'image. Ce qui l'établit, c'est la mesure directe —
   le stabilisateur d'axe est fermé par produit et son profil par ordres est $[1,2,2,2,2,2,4,4]$,
   signature de $D_8$ ($Q_8$ aurait six éléments d'ordre $4$).
2. $D_8$ a cinq éléments d'ordre $2$ et deux d'ordre $4$ ; $Q_8$ aurait six éléments d'ordre $4$
   et un seul d'ordre $2$. Les cinq classes de conjugaison (neutre, demi-tour central, deux quarts
   de tour, réflexions diagonales, réflexions hors-diagonales) correspondent aux cinq irréductibles
   $A_1, A_2, B_1, B_2, E$.
3. C'est l'algèbre de Lie engendrée par les trois générateurs du chapitre 55, dont les crochets
   littéraux donnent les constantes $(-1,-1,-1)$ ; une **seule** ré-orientation de signe sur les
   générateurs les ramène au symbole alterné $\epsilon_{abc}$ exact — l'algèbre est $\mathfrak{so}(3)$,
   écrit dans une base mal orientée.
4. Parce que $\mathrm{ad}_{T_3}$ a la structure $J$ avec $J^3 = -J$ : la série entière se referme en
   $I + \sin\theta\,J + (1-\cos\theta)J^2$, et au quart de tour $\sin = 1$, $\cos = 0$, d'où
   $I + J + J^2$. Sa puissance quatrième vaut $I_3$ : le quart de tour est d'ordre $4$ sur les
   directions, alors que le tour complet vaut $-I$ au bout qubit.

</details>

## Pour aller plus loin

*Chaînes-11 — le groupe de Lorentz et sa signature*
est le carnet que la conclusion annonce : « the next chapter joins them to the order coordinate in
the Lorentz group » — les rotations dérivées ici y rencontrent la coordonnée d'ordre du chapitre 52.

En amont, [Chaînes-09 — la coordonnée d'ordre](Chaines-09-Coordonnee-Ordre-Chaine-Serie-Python.ipynb)
est le carnet dont la conclusion annonçait celui-ci (« with order read as a count, the next chapter
turns to the rotations »), et [Chaînes-08 — les corollaires à leurs racines](Chaines-08-Corollaires-Quantique-Racines-CSharp.ipynb)
en fournit les corollaires quantiques.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
reste le contraste de méthode : là où une conservation se dérive d'une symétrie continue, le groupe
décrit ici est **fini**, et c'est son action dérivée — non postulée — qui fait le théorème 53.1.

Le jumeau Python de ce carnet, [Chaines-10 — le groupe des rotations](Chaines-10-Groupe-Rotations-Automorphismes-Directions-Python.ipynb),
rejoue le MÊME fil pédagogique en Python.

> Carnet issu de l'EPIC #26534.